# سبق 07 - پلاننگ ڈیزائن پیٹرن

یہ نوٹ بُک مائیکروسافٹ ایجنٹ فریم ورک کا استعمال کرتے ہوئے AI ایجنٹس کے لیے **پلاننگ ڈیزائن پیٹرن** دکھاتا ہے۔
آپ سیکھیں گے کہ کس طرح پیچیدہ سفر کی درخواستوں کو منظم ذیلی کاموں میں تقسیم کیا جائے، انہیں ماہر ایجنٹس کو تفویض کیا جائے،
اور حاصل شدہ منصوبہ کو نافذ کیا جائے — یہ سب پائیڈینٹک ماڈلز سے چلنے والے منظم آؤٹ پٹ کا استعمال کرتے ہوئے۔


## سیٹ اپ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ٹاسک کو تقسیم کرنا

ٹاسک کو تقسیم کرنا منصوبہ بندی کے ڈیزائن پیٹرن کا بنیادی حصہ ہے۔ ایک ہی ایجنٹ سے پیچیدہ درخواست کو شروع سے آخر تک ہینڈل کرنے کی بجائے، ہم مسئلہ کو چھوٹے، واضح طور پر متعین **ذیلی کاموں** میں تقسیم کرتے ہیں۔ ہر ذیلی کام کو ایک ماہر ایجنٹ (مثلاً، پروازیں، ہوٹلز، سرگرمیاں) کو واضح ترجیحات اور انحصار کی ترتیب کے ساتھ تفویض کیا جاتا ہے۔

یہ طریقہ کئی فوائد فراہم کرتا ہے:
- **وضاحت**: ہر ذیلی کام کی ایک واحد ذمہ داری ہوتی ہے۔
- **متوازی عمل**: آزاد ذیلی کام بیک وقت چل سکتے ہیں۔
- **قابل بھروسہ**: ناکامیاں انفرادی ذیلی کام تک محدود رہتی ہیں۔
- **بجٹ کی نگرانی**: لاگتوں کا تخمینہ ہر ذیلی کام کے لیے لگایا جاتا ہے اور جمع کیا جاتا ہے۔


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ساختہ شدہ آؤٹ پٹ کے ساتھ منصوبہ بند ایجنٹ بنانا

منصوبہ بند ایجنٹ ایک **فرنٹ ڈیسک کوآرڈینیٹر** کے طور پر کام کرتا ہے۔ ایک اعلیٰ سطحی سفر کی درخواست کو مدنظر رکھتے ہوئے یہ ایک ساختہ `TravelPlan` تیار کرتا ہے — درخواست کو ذیلی کاموں میں تقسیم کرتے ہوئے، ترجیحات مقرر کرتا ہے، اور انحصار کی نشاندہی کرتا ہے تاکہ ایک کانسیئر یا عمل درآمد کرنے والی سطح کام کو انجام دے سکے۔


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## ماہر آلات کے ساتھ منصوبہ کو عملی جامہ پہنانا

جب فرنٹ ڈیسک ایجنٹ نے ایک منظم منصوبہ تیار کر لیا ہو، تو **کنسیئرج ایجنٹ** اسے نافذ کرتا ہے۔  
ہر ماہر آلہ ذیلی کام کی ایک قسم (پروازیں، ہوٹل، سرگرمیاں) سنبھالتا ہے۔ کنسیئرج منصوبے کے ذیلی کاموں کو ان کے انحصاری ترتیب میں دہرائے گا اور ہر ایک کو مناسب آلے کو بھیجے گا۔


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## خلاصہ

اس سبق میں آپ نے AI ایجنٹس کے لیے **پلاننگ ڈیزائن پیٹرن** سیکھا:

1. **کام کی تقسیم** — ایک فرنٹ ڈیسک پلاننگ ایجنٹ پیچیدہ سفری درخواست کو Pydantic ماڈلز کا استعمال کرتے ہوئے منظم ذیلی کاموں میں تقسیم کرتا ہے، اور ہر ایک کو ترجیحات اور انحصاریوں کے ساتھ ماہر ایجنٹ کو تفویض کرتا ہے۔
2. **منظم آؤٹ پٹ** — `response_format` فراہم کر کے ایجنٹ آزاد متن کی بجائے ایک تصدیق شدہ `TravelPlan` آبجیکٹ واپس کرتا ہے، جو نیچے کی پراسیسنگ کو قابل اعتماد بناتا ہے۔
3. **منصوبہ بندی کا نفاذ** — ایک کانسیئر ایجنٹ ذیلی کاموں کے ذریعے ماہر آلات (`book_flight`, `reserve_hotel`, `book_activity`) استعمال کرتے ہوئے منصوبہ عملی جامہ پہنا کر نتائج رپورٹ کرتا ہے۔

یہ نمونہ *کیا کرنا ہے* (پلاننگ) کو *کیسے کرنا ہے* (نفاذ) سے الگ کرتا ہے، جس سے ایجنٹس زیادہ ماڈیولر، آزمائشی، اور آسانی سے توسیع پذیر بن جاتے ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**اخطار**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہِ کرم یہ بات ذہن میں رکھیں کہ خودکار ترجمے میں غلطیاں یا بے ضابطگیاں ہو سکتی ہیں۔ اصل دستاویز اپنی مادری زبان میں معتبر ماخذ سمجھی جانی چاہیے۔ اہم معلومات کے لیے پیشہ ورانہ انسانی ترجمہ تجویز کیا جاتا ہے۔ اس ترجمے کے استعمال سے ہونے والی کسی بھی غلط فہمی یا غلط تشریح کے لیے ہم ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
